In [ ]:
# ================================== КОНФИГ ==================================
# Ветка воспроизведения:
#   "end2end"       — полный стек из src/end2end/ обучается с нуля на пяти CSV
#                     соревнования. Внешних артефактов нет.
#  					Часы обучения, submit немного отличается, но лучший и оптимизирован по времени  §7.
#   "locked_replay" — побайтовый повтор best_submit: обе residual-модели
#                     обучаются здесь, базовый ансамбль берётся готовым, код ансамбля невосстановим (dead ssd), §8.
SOLUTION = "end2end"

import gc, json, os, re, shutil, subprocess, sys, time, warnings
from pathlib import Path

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 60)
plt.rcParams.update({"figure.figsize": (11, 3.2), "axes.grid": True, "grid.alpha": 0.3})

# --- где лежит код и данные ------------------------------------------------
REPO_URL, REPO_SHA = "https://github.com/Den221B/antifraud.git", None
KAGGLE = Path("/kaggle/input").exists()

def locate_repo():
    """Код ищем по порядку: рядом -> приложенный датасет -> git clone."""
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "src" / "pipeline.py").exists():
            return candidate
    if KAGGLE:
        for found in sorted(Path("/kaggle/input").rglob("pipeline.py")):
            if found.parent.name == "src":
                return found.parent.parent
        if REPO_URL and "<user>" not in REPO_URL:    # нужен включённый Internet
            target = Path("/kaggle/working/solution")
            if not target.exists():
                subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(target)], check=True)
                if REPO_SHA:
                    subprocess.run(["git", "-C", str(target), "checkout", REPO_SHA], check=True)
            return target
    raise FileNotFoundError(
        "не найден src/pipeline.py: приложите датасет с папками src/ и artifacts/ "
        "либо заполните REPO_URL и включите Internet")

REPO = locate_repo()
sys.path.insert(0, str(REPO))

def locate_data():
    """Ищем рядом, затем во всём /kaggle/input — данные соревнования лежат глубже."""
    local = REPO / "data"
    if (local / "train_transaction.csv").exists():
        return local
    if KAGGLE:
        for found in sorted(Path("/kaggle/input").rglob("train_transaction.csv")):
            if (found.parent / "test_transaction.csv").exists():
                return found.parent
    available = ([str(x) for x in sorted(Path("/kaggle/input").rglob("*.csv"))][:20]
                 if KAGGLE else [])
    raise FileNotFoundError(
        "не найден train_transaction.csv. Подключите данные соревнования "
        "(Add Input -> Competitions). Сейчас в /kaggle/input видно: "
        + (", ".join(available) if available else "ничего"))

DATA_DIR   = locate_data()
ARTIFACTS  = REPO / "artifacts"          # база, cold-маска, эталонный сабмит
MODELS_B2B = REPO / "models" / "byte2byte"   # веса двух LightGBM, 71 МБ, вне git
MODELS_E2E = REPO / "models" / "end2end"     # что оставляет после себя стек §7
for folder in (MODELS_B2B, MODELS_E2E):
    folder.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = Path("/kaggle/working" if KAGGLE else REPO / "notebook_output"); OUTPUT_DIR.mkdir(exist_ok=True)

# --- переключатели прогона (полностью ~50 мин, пик памяти ~6 ГБ) -----------
RUN_EDA        = True    # §2-3: разведка и проверочные эксперименты
TRAIN_MODELS   = True    # False -> веса из models/byte2byte, а если их нет —
                         #          готовые предсказания artifacts/pred_*.npy
VALIDATE_BLEND = True    # §6: проверка рецепта смешивания на нетронутом фолде
N_JOBS         = -1
# Ветка end2end (§7) переключателя «не учить» не имеет: она всегда учится с нуля.

# --- модели ----------------------------------------------------------------
LGB_PARAMS = {
    "n_estimators": 750, "objective": "binary", "metric": "auc", "boosting_type": "gbdt",
    "num_leaves": 47, "learning_rate": 0.035, "min_child_samples": 90,
    "subsample": 0.78, "subsample_freq": 1, "colsample_bytree": 0.78,
    "reg_alpha": 0.75, "reg_lambda": 10.0, "max_bin": 255, "max_depth": -1,
    "extra_trees": True, "n_jobs": N_JOBS, "verbosity": -1, "force_col_wise": True,
}
MODEL_SPECS = {
    "vblock":     {"view": "vblock",     "seed": 9203,  "file": "vblock_dynamics_final.txt"},
    "structured": {"view": "structured", "seed": 13303, "file": "vblock_structured_final.txt"},
}
# заменитель базы для проверки рецепта смешивания на holdout-фолде (§6).
# В финальные сабмиты не входит: в §7 базу строит стек end2end, в §8 — готовый файл.
BASE_LGB_PARAMS = {**LGB_PARAMS, "n_estimators": 800, "num_leaves": 63,
                   "learning_rate": 0.04, "extra_trees": False}
BASE_SEED = 4242

# --- веса смешивания, зафиксированы temporal-валидацией до прогона ----------
VBLOCK_COLD_WEIGHT     = 0.15 * 0.75   # OOF-вес 0.15 x shrinkage 0.75
STRUCTURED_COLD_WEIGHT = 0.025
UID_CONSISTENCY_WEIGHT = 0.50

print(f"ветка: {SOLUTION} | репозиторий: {REPO} | данные: {DATA_DIR}")
print("lightgbm", lgb.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

In [ ]:
from src.common import (DAY_SECONDS, HOLDOUT_FOLD, SEGMENT_COLUMNS, TARGET,
                        TEMPORAL_FOLDS, TOP_V_COLUMNS, V_BLOCKS, rank_prediction)
from src.features_blocks import row_vblock_features
from src.pipeline import (apply_residuals, assign_segments, build_all_features,
                          uid_max_consistency)

print(f"V-блоков {len(V_BLOCKS)} | избранных V {len(TOP_V_COLUMNS)} | окон валидации {len(TEMPORAL_FOLDS)}")

<a id="toc"></a>
# Антифрод: от данных до сабмита

Предсказываем `isFraud`. Метрика ROC AUC — важен порядок строк по риску, а не калибровка.

| Раздел | О чём | Можно пропустить |
|---|---|---|
| [1. Данные](#s1) | что лежит внутри, время, пропуски, скрытый пользователь | — |
| [2. Валидация](#s2) | почему KFold врёт и на чём проверяем решения | `RUN_EDA=False` |
| [3. Проверка гипотез](#s3) | сырые V против свёрнутых, откуда 30 избранных | `RUN_EDA=False` |
| [4. Признаки](#s4) | сборка 712 и 1046 признаков | — |
| [5. Модели](#s5) | две LightGBM, важность признаков | — |
| [6. Рецепт смешивания](#s6) | проверка весов на нетронутом фолде | `VALIDATE_BLEND=False` |
| [7. Ветка end2end](#s7) | полный стек с нуля, без внешних артефактов | `SOLUTION` |
| [8. Ветка locked_replay](#s8) | побайтовый повтор отправленного файла | `SOLUTION` |
| [9. Сверка и границы](#s9) | что совпало, что утеряно | — |

Разделы 2–3 — исследование: они обосновывают константы из конфига, но на результат
не влияют и отключаются флагом. Кто пришёл за воспроизводимостью — сразу в
[§4](#s4), [§7](#s7) и [§9](#s9).

Две ветки отвечают на два разных вопроса. `end2end` — «покажите работающее решение»:
всё обучается с нуля на данных соревнования. `locked_replay` — «докажите, что тот
самый submit ваш»: он воспроизводится побайтово, но опирается на одно сохранившееся
предсказание базового ансамбля.

<a id="s1"></a>
## 1. Данные

In [ ]:
overview = []
for name in ("train_transaction", "train_identity", "test_transaction", "test_identity", "sample_submission"):
    path = DATA_DIR / f"{name}.csv"
    overview.append({"файл": name, "строк": sum(1 for _ in path.open("rb")) - 1,
                     "колонок": pd.read_csv(path, nrows=1).shape[1],
                     "МБ": round(path.stat().st_size / 1024**2, 1)})
pd.DataFrame(overview)

In [ ]:
meta_cols = ["TransactionID", "TransactionDT", "TransactionAmt", "ProductCD", TARGET]
train_meta = pd.read_csv(DATA_DIR / "train_transaction.csv", usecols=meta_cols)
test_meta = pd.read_csv(DATA_DIR / "test_transaction.csv", usecols=[c for c in meta_cols if c != TARGET])
train_meta["day"] = train_meta.TransactionDT / DAY_SECONDS
test_meta["day"] = test_meta.TransactionDT / DAY_SECONDS

print(f"train {len(train_meta):>7,}  дни {train_meta.day.min():6.1f}-{train_meta.day.max():6.1f}  фрод {train_meta[TARGET].mean():.2%}")
print(f"test  {len(test_meta):>7,}  дни {test_meta.day.min():6.1f}-{test_meta.day.max():6.1f}")
print(f"разрыв между train и test: {test_meta.day.min() - train_meta.day.max():.1f} дней данных нет")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.2))
axes[0].hist(train_meta.day, bins=np.arange(0, 185), color="#3b76af", label="train")
axes[0].hist(test_meta.day, bins=np.arange(0, 185), color="#d1633f", label="test")
axes[0].axvspan(train_meta.day.max(), test_meta.day.min(), color="grey", alpha=0.3)
axes[0].set_title("транзакции по дням (серое — разрыв)"); axes[0].legend(); axes[0].set_xlabel("день")

weekly = train_meta.assign(w=(train_meta.day // 7).astype(int)).groupby("w")[TARGET].mean() * 100
axes[1].plot(weekly.index, weekly.values, marker="o", color="#b03a2e")
axes[1].axhline(train_meta[TARGET].mean() * 100, ls="--", color="grey")
axes[1].set_title(f"доля фрода по неделям, % ({weekly.min():.1f}-{weekly.max():.1f})"); axes[1].set_xlabel("неделя")
plt.tight_layout(); plt.show()

Train и test разделены во времени: случайный KFold обучал бы модель на будущем. Доля фрода плавает в 2.4 раза — признаки по таргету поедут на сдвиге. Обе вещи проверяем в §2.

In [ ]:
all_cols = pd.read_csv(DATA_DIR / "train_transaction.csv", nrows=1).columns.tolist()
groups = {"C1-C14": [c for c in all_cols if re.fullmatch(r"C\d+", c)],
          "D1-D15": [c for c in all_cols if re.fullmatch(r"D\d+", c)],
          "M1-M9": [c for c in all_cols if re.fullmatch(r"M\d+", c)],
          "V1-V339": [c for c in all_cols if re.fullmatch(r"V\d+", c)],
          "card/addr/dist": [c for c in all_cols if c.startswith(("card", "addr", "dist"))],
          "email": [c for c in all_cols if c.endswith("emaildomain")]}
nan_sample = pd.read_csv(DATA_DIR / "train_transaction.csv", usecols=sum(groups.values(), []))
pd.DataFrame([{"группа": n, "колонок": len(c),
               "средняя доля пропусков": round(nan_sample[c].isna().mean().mean(), 3)}
              for n, c in groups.items()])

In [ ]:
v_missing = nan_sample[groups["V1-V339"]].isna()
rate = v_missing.mean().round(5)
print(f"339 V-колонок дают всего {rate.nunique()} разных долей пропусков -> заполняются пачками")

rows = []
for a, b in V_BLOCKS:
    block = v_missing[[f"V{i}" for i in range(a, b + 1)]]
    filled = block.sum(axis=1)
    rows.append({"блок": f"V{a}-V{b}", "колонок": b - a + 1,
                 "пропуски от": round(block.mean().min(), 3), "до": round(block.mean().max(), 3),
                 "строк «всё или ничего»": round(float(filled.isin([0, block.shape[1]]).mean()), 3)})
pd.DataFrame(rows)

Внутри каждого из 11 блоков колонки пусты/заполнены вместе — блок несёт примерно одну единицу информации. Гипотеза: свернуть блок в сводки вместо 339 сырых колонок. Проверка в §2.3.

In [ ]:
uid_frame = pd.read_csv(DATA_DIR / "train_transaction.csv",
                        usecols=["TransactionDT", "card1", "addr1", "D1", "P_emaildomain", TARGET])
uid_frame["origin_day"] = (uid_frame.TransactionDT / DAY_SECONDS - uid_frame.D1).round()
valid = uid_frame[["card1", "addr1", "origin_day"]].notna().all(axis=1)
uid_frame["uid"] = (uid_frame[["card1", "addr1", "origin_day", "P_emaildomain"]]
                    .astype("string").fillna("?").agg("|".join, axis=1).where(valid))


def contagion(labels, target):
    frame = pd.DataFrame({"g": labels, "y": target}).dropna(subset=["g"])
    group = frame.groupby("g")["y"]
    others = (group.transform("sum") - frame.y) > 0
    multi = group.transform("size") > 1
    return float(frame.loc[multi & others, "y"].mean()), float(frame.loc[multi & ~others, "y"].mean())


rng = np.random.default_rng(0)
shuffled = uid_frame.uid.copy()
known = shuffled.notna()
shuffled[known] = rng.permutation(shuffled[known].to_numpy())
real, fake = contagion(uid_frame.uid, uid_frame[TARGET]), contagion(shuffled, uid_frame[TARGET])

sizes = uid_frame.uid.value_counts()
print(f"origin_day = round(TransactionDT/86400 - D1); ключ card1|addr1|origin_day|P_emaildomain")
print(f"валидных строк {valid.sum():,}/{len(uid_frame):,}, групп {len(sizes):,}, из них >1 строки {(sizes > 1).sum():,}")
print(f"базовая доля фрода               {uid_frame[TARGET].mean():.2%}")
print(f"P(фрод | сосед по UID фродовый)  {real[0]:.2%}   на перемешанных UID {fake[0]:.2%}")
print(f"P(фрод | сосед чистый)           {real[1]:.2%}   на перемешанных UID {fake[1]:.2%}")

Фрод «заразен» внутри UID и не заразен на перемешанных группах — ключ действительно склеивает связанные строки. Значит агрегаты по пользователю будут сильными, но склейку надо ограничивать по размеру.

In [ ]:
train_segment_source = pd.read_csv(DATA_DIR / "train_transaction.csv", usecols=list(SEGMENT_COLUMNS))
test_segment_source = pd.read_csv(DATA_DIR / "test_transaction.csv", usecols=list(SEGMENT_COLUMNS))
segments_train_only = assign_segments(train_segment_source, test_segment_source)
pd.Series(segments_train_only).value_counts().rename("строк test").to_frame().assign(
    доля=lambda f: (f["строк test"] / len(segments_train_only)).round(3))

Две трети test — пользователи без истории. Исторические признаки там пустые, значит качество по сегментам будет разным; проверяем в §2.2.

<a id="s2"></a>
## 2. Валидация и проверка гипотез
Простая модель на сырых колонках — цель не максимум AUC, а решение: чему верить и что оставить.

In [ ]:
EXPERIMENT_PARAMS = dict(n_estimators=300, objective="binary", learning_rate=0.05, num_leaves=63,
                         min_child_samples=60, colsample_bytree=0.8, subsample=0.8, subsample_freq=1,
                         reg_lambda=5.0, n_jobs=N_JOBS, verbosity=-1, random_state=42)


def fit_predict(X_train, y_train, X_valid):
    return lgb.LGBMClassifier(**EXPERIMENT_PARAMS).fit(X_train, y_train).predict_proba(X_valid)[:, 1]


simple_cat = ["ProductCD", "card4", "card6", "P_emaildomain", "R_emaildomain"] + [f"M{i}" for i in range(1, 10)]
simple_num = (["TransactionAmt", "card1", "card2", "card3", "card5", "addr1", "addr2", "dist1", "dist2"]
              + [f"C{i}" for i in range(1, 15)] + [f"D{i}" for i in range(1, 16)])
exp = pd.read_csv(DATA_DIR / "train_transaction.csv", usecols=["TransactionDT", TARGET] + simple_num + simple_cat)
exp_day, exp_y = exp.TransactionDT / DAY_SECONDS, exp[TARGET].to_numpy()
X_simple = exp[simple_num].astype("float32").copy()
for column in simple_cat:
    X_simple[column] = pd.factorize(exp[column].astype("string").fillna("?"))[0].astype("int32")
X_simple["hour"] = ((exp.TransactionDT // 3600) % 24).astype("int16")
X_simple["dayofweek"] = ((exp.TransactionDT // 86400) % 7).astype("int16")
print("простая матрица:", X_simple.shape)

### 2.1. Случайный KFold против временных окон

In [ ]:
temporal_predictions = {}
if RUN_EDA:
    scores = []
    for train_index, valid_index in StratifiedKFold(5, shuffle=True, random_state=0).split(X_simple, exp_y):
        prediction = fit_predict(X_simple.iloc[train_index], exp_y[train_index], X_simple.iloc[valid_index])
        scores.append(roc_auc_score(exp_y[valid_index], prediction))
    kfold_auc = float(np.mean(scores))

    rows = []
    for fold, (train_end, valid_start, valid_end) in enumerate(TEMPORAL_FOLDS):
        train_mask = (exp_day < train_end).to_numpy()
        valid_mask = ((exp_day >= valid_start) & (exp_day < valid_end)).to_numpy()
        prediction = fit_predict(X_simple[train_mask], exp_y[train_mask], X_simple[valid_mask])
        temporal_predictions[fold] = (valid_mask, prediction)
        rows.append({"fold": fold, "train дни": f"<{train_end}", "valid дни": f"{valid_start}-{valid_end}",
                     "train строк": int(train_mask.sum()), "valid строк": int(valid_mask.sum()),
                     "AUC": round(roc_auc_score(exp_y[valid_mask], prediction), 4)})
    temporal_table = pd.DataFrame(rows)
    print(f"случайный 5-fold AUC {kfold_auc:.4f}   временная валидация AUC {temporal_table.AUC.mean():.4f}"
          f"   разрыв {kfold_auc - temporal_table.AUC.mean():+.4f}")
    display(temporal_table)

KFold завышает качество на ~0.07 AUC. Все дальнейшие решения принимаются по временным фолдам, последний держим нетронутым holdout.

### 2.2. Сегменты

In [ ]:
if RUN_EDA:
    rows = []
    for fold, (train_end, _, _) in enumerate(TEMPORAL_FOLDS):
        valid_mask, prediction = temporal_predictions[fold]
        history = train_segment_source[train_segment_source.TransactionDT / DAY_SECONDS < train_end]
        segments = assign_segments(history, train_segment_source[valid_mask])
        for name in ("strict", "partial", "cold"):
            part = segments == name
            if part.sum() < 200 or exp_y[valid_mask][part].sum() == 0:
                continue
            rows.append({"fold": fold, "сегмент": name, "строк": int(part.sum()),
                         "фрод %": round(100 * exp_y[valid_mask][part].mean(), 2),
                         "AUC": round(roc_auc_score(exp_y[valid_mask][part], prediction[part]), 4)})
    segment_table = pd.DataFrame(rows)
    display(segment_table.pivot(index="fold", columns="сегмент", values="AUC"))

Разброс AUC между сегментами доходит до 0.17 — добавка, полезная на одном сегменте, легко съедается потерей на другом. Отсюда правило: residual-модели подмешиваем посегментно, а не всем сразу.

<a id="s3"></a>
## 3. Проверка гипотез

Сырые V против свёрнутых блоков

In [ ]:
if RUN_EDA:
    v_raw = pd.read_csv(DATA_DIR / "train_transaction.csv", usecols=[f"V{i}" for i in range(1, 340)])
    train_end, valid_start, valid_end = TEMPORAL_FOLDS[HOLDOUT_FOLD]
    train_mask = (exp_day < train_end).to_numpy()
    valid_mask = ((exp_day >= valid_start) & (exp_day < valid_end)).to_numpy()
    segments = assign_segments(
        train_segment_source[train_segment_source.TransactionDT / DAY_SECONDS < train_end],
        train_segment_source[valid_mask])
    y_valid = exp_y[valid_mask]

    rows = []
    for name, matrix in (("без V", X_simple),
                         ("+ сырые V1-V339", pd.concat([X_simple, v_raw], axis=1)),
                         ("+ свёрнутые V-блоки", pd.concat([X_simple, row_vblock_features(v_raw)], axis=1))):
        prediction = fit_predict(matrix[train_mask], exp_y[train_mask], matrix[valid_mask])
        row = {"набор": name, "признаков": matrix.shape[1], "AUC": round(roc_auc_score(y_valid, prediction), 4)}
        for segment in ("strict", "cold"):
            row[f"AUC {segment}"] = round(roc_auc_score(y_valid[segments == segment],
                                                        prediction[segments == segment]), 4)
        rows.append(row)
        del matrix; gc.collect()
    display(pd.DataFrame(rows))

Сырые `V` ухудшают и общий AUC (0.8707 против 0.8730 без них), и `cold` (0.8671 против 0.8698), выигрывая только на `strict`. Свёрнутые блоки лучше всех трёх вариантов по общему AUC и по `cold`. 339 разреженных коррелирующих колонок дают больше поводов переобучиться, чем сигнала: в финальный набор сырые `V` не идут, идут сводки.

In [ ]:
if RUN_EDA:
    gain = pd.Series(lgb.LGBMClassifier(**EXPERIMENT_PARAMS).fit(v_raw[train_mask], exp_y[train_mask])
                     .booster_.feature_importance("gain"), index=v_raw.columns)
    place = gain.rank(ascending=False).astype(int)[list(TOP_V_COLUMNS)].sort_values()
    print(f"TOP_V_COLUMNS из конфига: медиана места по gain {int(place.median())} из {len(gain)}, "
          f"худшее {int(place.max())}, в топ-100 попало {(place <= 100).sum()}/30")
    display(place.head(10).rename("место").to_frame().T)
    del v_raw; gc.collect()

Список из 30 `V` не случаен — верхние 10% по gain. Берём по ним агрегаты внутри пользователя, а не сырые значения (см. предыдущий эксперимент).

In [ ]:
for name in ("exp", "X_simple", "nan_sample", "v_missing", "uid_frame", "temporal_predictions",
             "train_meta", "test_meta", "shuffled", "gain", "sizes"):
    globals().pop(name, None)
_ = gc.collect()

<a id="s4"></a>
## 4. Признаки

Весь конвейер — в `src/`, здесь только вызов. Ни один признак не использует таргет:
счётчики фрода по истории пользователя не пережили временной сдвиг из §1.

| Модуль | Что делает |
|---|---|
| `features_row` | построчные признаки, UID-цепочки, частоты по train |
| `features_graph` | два графа пользователей, агрегаты по компонентам, поведение |
| `features_blocks` | свёртка V-блоков, суммы, устройство, распределения значений |
| `pipeline` | сборка двух наборов, сегментация, смешивание |

In [ ]:
data = build_all_features(DATA_DIR)
train, test, y = data["train"], data["test"], data["y"]
train_ids, test_ids = data["train_ids"], data["test_ids"]
VIEWS, categorical = data["views"], data["categorical"]
_ = gc.collect()

In [ ]:
print("граф 1:", {k: v for k, v in data["stats"]["graph"].items() if k != "keys"})
print("граф 2:", {k: v for k, v in data["stats"]["advanced_graph"].items() if k != "exact_keys"})
pd.DataFrame([{"группа": name, "признаков": len(columns)} for name, columns in data["groups"].items()])

Ограничитель компонент работает: ~137 тысяч «пользователей» на 495 тысяч строк, максимум 372 строки в компоненте — гигантских склеек нет.

<a id="s5"></a>
## 5. Residual-модели
LightGBM с одинаковыми параметрами на двух взглядах, разные сиды. `extra_trees` и `reg_lambda=10` против переобучения на редких категориях; фиксированные 750 деревьев без ранней остановки — иначе результат зависит от того, какой кусок попал в валидацию.

In [ ]:
def weights_path(spec):
    """Веса по 35 МБ в git не кладём: ищем их рядом, иначе работаем по предсказаниям."""
    return next((p for p in (MODELS_B2B / spec["file"], OUTPUT_DIR / spec["file"]) if p.exists()), None)


def fit_lgb(name):
    """-> (booster | None, предсказание на test). None означает «весов нет, взяли .npy»."""
    spec = MODEL_SPECS[name]
    features = VIEWS[spec["view"]]

    if not TRAIN_MODELS:
        path = weights_path(spec)
        if path is not None:
            booster = lgb.Booster(model_file=str(path))
            assert booster.feature_name() == features, f"{name}: порядок признаков разошёлся"
            print(f"{name}: веса из {path.parent.name}/{path.name}, {booster.num_trees()} деревьев")
            return booster, np.asarray(booster.predict(test[features]), dtype="float64")
        stored = ARTIFACTS / f"pred_{name}.npy"
        prediction = np.load(stored)
        assert len(prediction) == len(test_ids), f"{name}: длина {stored.name} не совпала с test"
        print(f"{name}: весов нет, взяты предсказания {stored.name} ({len(prediction):,} строк)")
        return None, prediction

    started = time.time()
    print(f"{name}: {len(features)} признаков, seed {spec['seed']}...", flush=True)
    model = lgb.LGBMClassifier(**LGB_PARAMS, random_state=spec["seed"]).fit(
        train[features], y, categorical_feature=[c for c in categorical if c in features],
        callbacks=[lgb.log_evaluation(0)])
    model.booster_.save_model(str(MODELS_B2B / spec["file"]))
    print(f"    {(time.time() - started) / 60:.1f} мин, веса -> models/byte2byte/{spec['file']}", flush=True)
    return model.booster_, np.asarray(model.booster_.predict(test[features]), dtype="float64")


fitted = {name: fit_lgb(name) for name in MODEL_SPECS}
boosters = {name: booster for name, (booster, _) in fitted.items() if booster is not None}
predictions = {name: prediction for name, (_, prediction) in fitted.items()}

In [ ]:
# важность считается по деревьям модели: если взяты готовые
# предсказания, самих моделей нет и график пропускается
if not boosters:
    print('модели не загружены — важность признаков недоступна')
else:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    family_rows = []
    for ax, (name, booster) in zip(axes, boosters.items()):
        importance = pd.Series(booster.feature_importance("gain"), index=booster.feature_name())
        top = importance.sort_values(ascending=False).head(15)
        ax.barh(top.index[::-1], top.to_numpy()[::-1], color="#3b76af")
        ax.set_title(f"{name}: топ-15 по gain", fontsize=10); ax.tick_params(axis="y", labelsize=7)
        family = pd.Series("поля транзакции", index=importance.index)
        for prefix, label in (("vblock_", "V-блоки"), ("wide_user_", "профиль (V-блок)"), ("user_", "профиль"),
                              ("adv_user_", "поведение"), ("behavior_", "распределения"), ("cooc_", "co-occurrence"),
                              ("amount_", "суммы"), ("calendar_", "календарь"), ("identity_", "устройство"),
                              ("uid_", "UID-цепочки")):
            family[importance.index.str.startswith(prefix)] = label
        share = importance.groupby(family).sum()
        family_rows.append((share / share.sum()).rename(name))
    plt.tight_layout(); plt.show()
    pd.concat(family_rows, axis=1).fillna(0).sort_values("structured", ascending=False).round(3)

Больше половины важности — пользователь и его поведение, а не поля самой транзакции. Свёрнутые V-блоки дают около трети важности первой модели при нуле сырых `V` в наборе.

<a id="s6"></a>
## 6. Рецепт смешивания на нетронутом фолде

Веса и правило «только cold» зафиксированы по OOF настоящей базы. Здесь проверяем их на holdout-фолде своими моделями: учимся на днях <60, проверяемся на 90–106.

In [ ]:
def fit_lgb_on(mask_train, mask_predict, view, seed, params):
    features = VIEWS[view]
    started = time.time()
    model = lgb.LGBMClassifier(**params, random_state=seed).fit(
        train.loc[mask_train, features], y.to_numpy()[mask_train],
        categorical_feature=[c for c in categorical if c in features],
        callbacks=[lgb.log_evaluation(0)])
    print(f"    {view}/{seed}: {(time.time() - started) / 60:.1f} мин", flush=True)
    frame = train.loc[mask_predict, features] if mask_predict is not None else test[features]
    return np.asarray(model.predict_proba(frame)[:, 1], dtype="float64")

In [ ]:
# нетронутый fold 3: учимся на днях <60, проверяемся на 90-106
fold_report = {}
if VALIDATE_BLEND:
    train_day = (train_segment_source.TransactionDT / DAY_SECONDS).to_numpy()
    train_end, valid_start, valid_end = TEMPORAL_FOLDS[HOLDOUT_FOLD]
    f_train = train_day < train_end
    f_valid = (train_day >= valid_start) & (train_day < valid_end)
    y_fold = y.to_numpy()[f_valid]
    f_cold = assign_segments(train_segment_source[f_train], train_segment_source[f_valid]) == "cold"
    print(f"fold {HOLDOUT_FOLD}: train {f_train.sum():,} -> valid {f_valid.sum():,}, "
          f"из них cold {f_cold.sum():,}", flush=True)

    fold_base = uid_max_consistency(train_segment_source[f_valid],
                                    fit_lgb_on(f_train, f_valid, "structured", BASE_SEED, BASE_LGB_PARAMS))
    fold_res = {name: fit_lgb_on(f_train, f_valid, spec["view"], spec["seed"], LGB_PARAMS)
                for name, spec in MODEL_SPECS.items()}

In [ ]:
if VALIDATE_BLEND:
    def fold_scores(label, prediction):
        return {"рецепт": label,
                "AUC": round(roc_auc_score(y_fold, prediction), 5),
                "AUC cold": round(roc_auc_score(y_fold[f_cold], prediction[f_cold]), 5),
                "AUC strict/partial": round(roc_auc_score(y_fold[~f_cold], prediction[~f_cold]), 5)}

    vb, st = fold_res["vblock"], fold_res["structured"]
    rows = [
        fold_scores("vblock-модель отдельно", rank_prediction(vb)),
        fold_scores("structured-модель отдельно", rank_prediction(st)),
        fold_scores("база (LGB structured + UID-consistency)", rank_prediction(fold_base)),
        fold_scores("+ vblock только cold", apply_residuals(fold_base, [(vb, VBLOCK_COLD_WEIGHT, f_cold)])),
        fold_scores("+ vblock всем строкам", apply_residuals(fold_base, [(vb, VBLOCK_COLD_WEIGHT, np.ones_like(f_cold))])),
        fold_scores("+ vblock только strict/partial", apply_residuals(fold_base, [(vb, VBLOCK_COLD_WEIGHT, ~f_cold)])),
        fold_scores("+ vblock + structured, cold (итог)",
                    apply_residuals(fold_base, [(vb, VBLOCK_COLD_WEIGHT, f_cold), (st, STRUCTURED_COLD_WEIGHT, f_cold)])),
    ]
    fold_report["recipes"] = rows
    display(pd.DataFrame(rows))

Три вещи видно сразу.

**Сегменты действительно разные.** База берёт `strict/partial` 0.925, а `cold` — 0.831.
Разрыв в 0.094 — это ровно та история, которой у холодных пользователей нет.

**Residual-модели сильны именно там, где база слаба.** По отдельности они дают на `cold`
0.872 и 0.874 против 0.831 у базы. Подмешивание к `cold` поднимает её до 0.840, оба
residual вместе — до 0.840 при общем 0.871.

**Правило «только cold» этой таблицей не доказывается — и не должно.** Подмешивание
всем строкам здесь даёт 0.870 против 0.870 у cold-only, то есть чуть лучше. Причина в
том, что база тут — заменитель, одна LightGBM вместо стека из 13 моделей, и на
`strict/partial` она слабее настоящей. У настоящей базы прирост на этом сегменте был
отрицательным, поэтому в зафиксированном рецепте подмешивание ограничено `cold`.
Веса и правило взяты из её OOF, а не подобраны здесь.

In [ ]:
if VALIDATE_BLEND:
    grid = []
    for weight in (0.0, 0.05, 0.1125, 0.2, 0.35, 0.5):
        prediction = apply_residuals(fold_base, [(fold_res["vblock"], weight, f_cold)])
        grid.append({"вес vblock": weight, "AUC": round(roc_auc_score(y_fold, prediction), 5),
                     "AUC cold": round(roc_auc_score(y_fold[f_cold], prediction[f_cold]), 5)})
    display(pd.DataFrame(grid))
    for name in ("fold_base", "fold_res", "vb", "st"):
        globals().pop(name, None)
    _ = gc.collect()

<a id="s7"></a>
## 7. Ветка `end2end`

Полный стек, обученный с нуля на пяти CSV соревнования.

Код лежит в `src/end2end/` четырнадцатью модулями. Каждый берёт данные из папки
рядом с собой, поэтому запуск идёт в отдельной рабочей директории: туда копируются
модули и подкладываются CSV. Ноутбук при этом не меняет свой рабочий каталог —
пути остальных разделов не съезжают.

| Стадия | Что обучается |
|---|---|
| 1 | независимый фундамент: CatBoost, LightGBM, XGBoost + нейронный стек по временным фолдам |
| 2 | CatBoost на 701-признаковом представлении пользователя, 3 сида, вес источника 0.25 |
| 3 | средние `C1–C14`, `D1–D15` и 30 избранных `V` по компоненте, вес residual 0.05 |
| 4 | клиентский мета-LightGBM поверх пяти источников + сегментная коррекция strict/partial/cold |

In [ ]:
E2E_DIR  = REPO / "src" / "end2end"
E2E_WORK = OUTPUT_DIR / "end2end_run"
E2E_SOURCE     = "submission_honest_client_profile_lgb.csv"   # что оставляет последняя стадия
E2E_SUBMISSION = "submission_honest_best.csv"                 # под этим именем уходит на сабмит
FORCE_RETRAIN  = False        # True -> стадии считают заново, игнорируя кэш

# Стадии в порядке запуска: (скрипт, аргументы, поддерживает ли --force)
E2E_STAGES = (
    # фундамент и клиентский temporal OOF
    ("train_clean_foundation.py",              (),                        False),
    ("train_boost3_neural_stack.py",           (),                        True),
    ("train_honest_advanced_catboost.py",      (),                        True),
    ("build_honest_advanced_cat_final.py",     (),                        True),
    ("train_honest_user_means_catboost.py",    ("--folds", "0,1,2"),      True),
    ("build_honest_user_means_final.py",       ("--seeds", "1729"),       True),
    ("train_honest_client_meta.py",            (),                        False),
    ("refine_honest_client_segments.py",       (),                        False),
    # представления V-блоков и структурных признаков
    ("prepare_honest_featureview_sources.py",  (),                        True),
    ("search_honest_featureview_meta.py",      (),                        False),
    ("finalize_honest_featureview_meta.py",    (),                        False),
    ("search_honest_fullrow_lgb.py",           (),                        False),
    # тяжёлый clean-v2 и XGB-magic
    ("clean_v2_pipeline.py",                   ("--mode", "all"),         True),
    ("search_honest_cleanv2_blend.py",         (),                        False),
    ("train_honest_xgb_magic.py",              (),                        False),
    ("finalize_honest_xgb_magic_blend.py",     (),                        False),
    ("train_honest_magic_heavy_stack.py",      (),                        False),
    # финальный клиентский residual
    ("train_honest_client_profile_lgb.py",     (),                        False),
)

FORBIDDEN = ("external_ieee", "external_test_labels_AUDIT_ONLY",
             "gap_transaction.csv", "gap_identity.csv")
# Хеши официальных входов: гарантируют, что считаем на той же версии данных
EXPECTED_SHA256 = {
    "train_transaction.csv": "d75f0cedaf354a750bc148372b63835efef53445abe017aad3d93dc2ebfc7da7",
    "train_identity.csv":    "8cece20966dc33be033ace5821ec183ea019a2d18008a9590e30a202d2da19d0",
    "test_transaction.csv":  "7116dd600f0f3b6fcc62292988a4997c02727265836baa816383dcf4202bdadd",
    "test_identity.csv":     "5ceabcf3453372247372399e3ff083f3727dd69675040235bc1ef00c27f05a81",
    "sample_submission.csv": "c2d4a3a53db995f95f7a10fe1785683303146d683ed71b8c396c8d31d4859e83",
}

modules = sorted(E2E_DIR.glob("*.py"))
violations = {p.name: [t for t in FORBIDDEN if t in p.read_text(encoding="utf-8")]
              for p in modules}
violations = {name: tokens for name, tokens in violations.items() if tokens}
print(f"модулей end2end: {len(modules)}, стадий: {len(E2E_STAGES)}")
print("проверка политики входов:", "нарушений нет" if not violations else violations)
assert not violations

In [ ]:
# предполётная проверка: без неё стадии упадут через несколько часов, а не сразу
import hashlib, importlib.util

E2E_REQUIRES = ("numpy", "pandas", "sklearn", "catboost", "lightgbm", "xgboost", "torch")
missing = [name for name in E2E_REQUIRES if importlib.util.find_spec(name) is None]
csv_ready = all((DATA_DIR / name).exists() for name in EXPECTED_SHA256)

# версия данных: стек валидировался ровно на этих файлах
digests = {}
if csv_ready:
    for name, expected in EXPECTED_SHA256.items():
        actual = hashlib.sha256((DATA_DIR / name).read_bytes()).hexdigest()
        digests[name] = actual == expected

print("зависимости:", "все на месте" if not missing else f"НЕ УСТАНОВЛЕНО: {missing}")
print("пять CSV соревнования:", "на месте" if csv_ready else "НЕ НАЙДЕНЫ")
print("сверка sha256 входов:", "совпали все" if digests and all(digests.values())
      else {k: v for k, v in digests.items() if not v} or "пропущена")

E2E_READY = not missing and csv_ready and all(digests.values())
if SOLUTION == "end2end" and not E2E_READY:
    raise RuntimeError(f"ветка end2end не готова: пакеты {missing or 'ок'}, "
                       f"данные {'ок' if csv_ready else 'не найдены'}, "
                       f"хеши {'ок' if digests and all(digests.values()) else 'разошлись'}")

In [ ]:
def prepare_e2e_workdir():
    """Модули ищут данные рядом с собой -> собираем изолированную рабочую папку."""
    E2E_WORK.mkdir(parents=True, exist_ok=True)
    for module in modules:
        shutil.copy2(module, E2E_WORK / module.name)
    for name in ("train_transaction.csv", "train_identity.csv", "test_transaction.csv",
                 "test_identity.csv", "sample_submission.csv"):
        destination = E2E_WORK / name
        if destination.exists() or destination.is_symlink():
            destination.unlink()
        try:
            destination.symlink_to((DATA_DIR / name).resolve())
        except OSError:                      # без прав на симлинки — копируем
            shutil.copy2(DATA_DIR / name, destination)
    return E2E_WORK


def run_e2e_stage(script, arguments, supports_force):
    command = [sys.executable, script, *arguments]
    if supports_force and FORCE_RETRAIN:
        command.append("--force")
    started = time.time()
    print(f"--> {' '.join(command[1:])}", flush=True)
    subprocess.run(command, cwd=E2E_WORK, check=True,
                   env={**os.environ, "PYTHONHASHSEED": "0"})
    assert not (E2E_WORK / "external_ieee").exists()   # стадия не создала запретный источник
    print(f"    {(time.time() - started) / 60:.1f} мин", flush=True)

In [ ]:
submission_end2end = None
if SOLUTION == "end2end":
    prepare_e2e_workdir()
    print("рабочая папка:", E2E_WORK, flush=True)
    started = time.time()
    for number, (script, arguments, supports_force) in enumerate(E2E_STAGES, 1):
        print(f"[{number}/{len(E2E_STAGES)}]", end=" ")
        run_e2e_stage(script, arguments, supports_force)
    print(f"всего {(time.time() - started) / 60:.1f} мин", flush=True)

    produced = E2E_WORK / E2E_SOURCE
    for model_file in sorted(E2E_WORK.rglob("*")):          # обученное -> models/end2end/
        if model_file.suffix in (".cbm", ".txt", ".pt", ".joblib") and model_file.is_file():
            shutil.copy2(model_file, MODELS_E2E / model_file.name)
    print(f"моделей сохранено в models/end2end: {len(list(MODELS_E2E.glob('*')))-1}", flush=True)
    submission_end2end = pd.read_csv(produced)
    sample = pd.read_csv(DATA_DIR / "sample_submission.csv")
    assert submission_end2end["TransactionID"].equals(sample["TransactionID"])
    assert submission_end2end["TransactionID"].is_unique
    assert submission_end2end[TARGET].between(0.0, 1.0).all()
    assert submission_end2end[TARGET].notna().all()
    submission_end2end.to_csv(OUTPUT_DIR / "submission_end2end.csv", index=False)
    print("сохранено:", OUTPUT_DIR / "submission_end2end.csv")

In [ ]:
e2e_summary = {}
if SOLUTION == "end2end":
    reports = {"user_means": "honest_user_means_catboost/final_report.json",
               "client_meta": "honest_client_meta/report.json",
               "segments": "honest_client_segments/report.json",
               "client_profile": "honest_client_profile_lgb/report.json"}
    loaded = {name: json.loads((E2E_WORK / path).read_text(encoding="utf-8"))
              for name, path in reports.items() if (E2E_WORK / path).exists()}
    profile = loaded.get("client_profile", {})
    e2e_summary = {
        "stages": len(E2E_STAGES),
        "modules": len(modules),
        "temporal_dev_auc": profile.get("dev_auc"),
        "temporal_lock_auc": profile.get("lock_auc"),
        "selected_residual": profile.get("selected"),
        "accepted": profile.get("accepted"),
        "client_meta_lock_auc": loaded.get("client_meta", {}).get("scores", {}).get("lock_final_auc"),
        "segment_lock_auc": loaded.get("segments", {}).get("segment_candidate_lock_auc"),
    }
    submitted = pd.read_csv(ARTIFACTS / "submission_best_solution.csv")
    e2e_summary["spearman_with_submitted"] = round(float(
        pd.Series(submission_end2end[TARGET]).corr(pd.Series(submitted[TARGET]), method="spearman")), 5)
    print(json.dumps(e2e_summary, indent=2, ensure_ascii=False))

<a id="s8"></a>
## 8. Ветка `locked_replay`

Побайтовое воспроизведение файла, отправленного в лидерборд. Из `artifacts/` берутся
ровно две вещи:

| Файл | Зачем | Почему не считается здесь |
|---|---|---|
| `base_submission_segmented.csv` | предсказание базового ансамбля — первый член формулы | утеряны промежуточные артефакты, см. §9 |
| `cold_mask.npy` | маска cold для test, 129 736 значений  | утеряны промежуточные артефакты, см. §9 |
Всё остальное — признаки, обе модели, смешивание — считается из сырых данных.

In [ ]:
submission_locked = None
if SOLUTION == "locked_replay":
    base_segmented = pd.read_csv(MODELS_B2B / "base_submission_segmented.csv")
    cold = np.load(MODELS_B2B / "cold_mask.npy")
    assert np.array_equal(base_segmented.TransactionID.to_numpy(), test_ids.to_numpy())
    assert len(cold) == len(test_ids)

    locked = json.loads((ARTIFACTS / "locked_run.json").read_text(encoding="utf-8"))
    print(f"cold {int(cold.sum()):,} / {len(cold):,}   в зафиксированном запуске "
          f"{locked['segments']['cold']:,}   совпало: {int(cold.sum()) == locked['segments']['cold']}")
    print(f"для сравнения, cold по одному train: {int((segments_train_only == 'cold').sum()):,}")

    final_locked = apply_residuals(base_segmented[TARGET], [
        (predictions["vblock"], VBLOCK_COLD_WEIGHT, cold),
        (predictions["structured"], STRUCTURED_COLD_WEIGHT, cold)])
    submission_locked = pd.DataFrame({"TransactionID": test_ids, TARGET: final_locked})
    submission_locked.to_csv(OUTPUT_DIR / "submission.csv", index=False)
    print("сохранено:", OUTPUT_DIR / "submission.csv")

In [ ]:
if SOLUTION == "locked_replay":
    base_rank_all = rank_prediction(base_segmented[TARGET])
    delta = final_locked - base_rank_all
    display(pd.DataFrame({
        "сегмент": ["cold", "strict/partial"], "строк": [int(cold.sum()), int((~cold).sum())],
        "Спирмен с базой": [round(float(pd.Series(base_rank_all[m]).corr(
            pd.Series(final_locked[m]), method="spearman")), 5) for m in (cold, ~cold)],
        "медиана |Δ ранга|": [round(float(np.median(np.abs(delta[m]))), 5) for m in (cold, ~cold)],
        "макс |Δ ранга|": [round(float(np.abs(delta[m]).max()), 5) for m in (cold, ~cold)]}))

<a id="s9"></a>
## 9. Сверка

In [ ]:
# Веса моделей (71 МБ) в репозиторий не входят, вместо них лежат их предсказания
# на test из зафиксированного прогона — этого достаточно, чтобы проверить,
# что переобученная здесь модель дала то же самое.
checks = []
for name, booster in boosters.items():
    path = ARTIFACTS / f"pred_{name}.npy"
    if not path.exists():
        continue
    reference = np.load(path)
    difference = np.abs(predictions[name] - reference)
    checks.append({"модель": name, "признаков": len(VIEWS[MODEL_SPECS[name]["view"]]),
                   "деревьев": booster.num_trees(),
                   "предсказания совпали": bool(np.array_equal(predictions[name], reference)),
                   "max |Δ|": float(difference.max())})
_ = gc.collect()
pd.DataFrame(checks) if checks else print("эталонные предсказания не приложены, сверка пропущена")

In [ ]:
submission = submission_locked if SOLUTION == "locked_replay" else submission_end2end
saved = pd.read_csv(OUTPUT_DIR / ("submission.csv" if SOLUTION == "locked_replay" else "submission_end2end.csv"))
best = pd.read_csv(ARTIFACTS / "submission_best_solution.csv")

report = {
    "branch": SOLUTION,
    "rows": len(saved),
    "identical_to_submitted": bool(np.array_equal(saved[TARGET].to_numpy(), best[TARGET].to_numpy())),
    "rows_differing": int((saved[TARGET].to_numpy() != best[TARGET].to_numpy()).sum()),
    "spearman_with_submitted": round(float(pd.Series(saved[TARGET]).corr(
        pd.Series(best[TARGET]), method="spearman")), 5),
    "features": {name: len(columns) for name, columns in VIEWS.items()},
    "models_trained_here": TRAIN_MODELS,
    "models_match_shipped": bool(all(c["предсказания совпали"] for c in checks)) if checks else None,
    "byte2byte_inputs": (["models/byte2byte/base_submission_segmented.csv",
                          "models/byte2byte/cold_mask.npy"]
                       if SOLUTION == "locked_replay" else []),
    "end2end": e2e_summary or None,
}
(OUTPUT_DIR / "run_report.json").write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(report, indent=2, ensure_ascii=False))

## Границы

**Что утеряно.** Базовый ансамбль исходного прогона — стек из 13 моделей с временной
валидацией, поверх него UID-consistency и сегментные residual. Пошагово он не
восстанавливается: нет промежуточных скриптов и их артифактов. Из-за командной работы часть скриптов были только у одного из участников, у одного из участников умер SSD, пришлось восставливать пайплайн, он назван e2e но сабмит не идентичен, хотя по метрике отличается незначительно.

Утеряны именно артефакты одного прогона, а не методика: ветка `end2end` (§7) обучает
полный стек с нуля на пяти CSV соревнования и даёт свой сабмит. Отсюда и разница между
ветками — это другой, заново собранный пайплайн, а не расхождение вычислений.

**Небольшое несоответствие** Около 5 прогонов этого кода дали байт в байт результаты. На каггле они отличают либо из-за ограничений среды, либо из-за отсутствия ресурсов. На машине с большим кол-вом ресурсов результаты должны быть 1 в 1.

**Проверено и отброшено:** цепочки V307 и мульти-счётчики (ломают ранние фолды),
causal velocity 1ч–30д (неустойчивый residual), байесовское target encoding по UID
(временной дрейф), все признаки одной моделью (шум и конкуренция за расщепления),
календарь без D и сумм (нет прироста).

[к оглавлению](#toc)